# 📉 Jour 4 — La Descente de Gradient : De NumPy aux Neurones Modernes avec PyTorch

Bienvenue dans ce quatrième module consacré aux **fondations algorithmiques et matérielles de l'IA moderne** !

Au cours du **Jour 3**, nous avons exploré le **Perceptron de Rosenblatt (1957)**. Nous avons découvert la beauté de sa règle géométrique, mais aussi les deux blocages fondamentaux qui l'ont empêché de donner naissance au Deep Learning :

1. **Le verrou de la non-différentiabilité (l'échelon binaire d'Heaviside)** :
   - *Comment le Perceptron de Rosenblatt apprenait-il ?*  
     Il utilisait une règle empirique locale : $\mathbf{w} \leftarrow \mathbf{w} + \eta (y_i - \hat{y}_i) \mathbf{x}_i$. Cette règle fonctionnait parce que le neurone était **unique et isolé** : le label attendu $y_i$ était directement comparé à la sortie de ce neurone.
   - *Pourquoi cette règle s'effondre-t-elle dès qu'on empile des neurones (réseaux profonds) ?*  
     Dans un réseau à plusieurs couches, les neurones des couches intermédiaires (les « couches cachées ») n'ont pas de label cible $y$ direct ! On ne sait pas ce qu'ils sont censés prédire individuellement. Pour savoir comment ajuster leurs poids, la seule solution mathématique est de propager l'erreur finale de la sortie vers l'arrière en calculant la dérivée partielle $\frac{\partial L}{\partial w}$ via la **règle de dérivation en chaîne** (*chain rule* / rétropropagation du gradient).
   - *Pourquoi l'échelon d'Heaviside tue cette propagation ?*  
     La fonction de décision d'Heaviside est un interrupteur brutal « tout-ou-rien » : elle vaut strictement $0$ ou $1$.  
     Sa dérivée (sa pente) est **plate comme une table** : $f'(z) = 0$ pour tout $z \ne 0$.  
     Si l'on modifie très légèrement un poids synaptique $w$ par un incrément infinitésimal $\epsilon$, la sortie $\hat{y}$ **ne varie pas d'un millimètre** ($\Delta \hat{y} = 0$). Comme la dérivée locale vaut zéro, dans la chaîne de multiplications de la rétropropagation, **tout est multiplié par zéro** ! L'algorithme d'optimisation est totalement aveugle : aucun signal de gradient ne peut circuler pour guider les couches cachées.
   - *La rupture moderne* :  
     Pour faire circuler un signal de gradient riche et continu, il faut remplacer l'interrupteur brutal par un **variateur progressif à courbe lisse** : la **fonction Sigmoïde** $\sigma(z) = \frac{1}{1 + e^{-z}}$. Dès lors, chaque micro-variation de poids entraîne une micro-variation mesurable de la sortie, offrant une pente informative (un gradient non nul) pour guider l'ensemble des neurones du réseau.

2. **Le verrou géométrique de la non-linéarité (le drame du XOR)** :
   Un neurone unique ne sait tracer qu'une droite (un hyperplan). Il est mathématiquement incapable de séparer des classes entrelacées comme le **OU Exclusif (XOR)**, ce qui avait conduit Marvin Minsky et Seymour Papert à condamner le modèle en 1969.

Pour débloquer ces deux verrous, il fallait réunir les deux clés de voûte de l'IA moderne : **des neurones lisses et différentiables** entraînés par **descente de gradient**, et **l'empilement en couches successives (MLP)**.

---

### 🎯 Objectifs de ce notebook :
1. **Comprendre la rupture de différentiabilité** :
   - Pourquoi le passage d'un seuil binaire brutal (Heaviside) à une courbe lisse (**Sigmoïde** $\sigma(z)$) débloque mathématiquement l'apprentissage des réseaux multicouches.
   - Remplacer le simple comptage discret d'erreurs par une fonction de coût continue et différentiable : la **Binary Cross-Entropy (Log-Loss)**.
2. **Visualiser la géométrie de la descente** : la métaphore de la cuvette dans le brouillard, les lignes de niveau et le rôle déterminant du taux d'apprentissage $\eta$ (*learning rate*).
3. **Construire la descente de gradient "from scratch" en pur NumPy** : dériver analytiquement le gradient $\nabla_{\mathbf{w}} L = \frac{1}{m} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$ et coder la classe vectorisée `NeuroneGradientNumPy`.
4. **Identifier le plafond de verre de NumPy** : dérivation manuelle impossible pour les réseaux multicouches, absence d'accélération matérielle GPU / Apple Silicon, absence de différentiation automatique.
5. **Entrer dans l'ère moderne avec PyTorch (`torch`)** :
   - Le tenseur optimisé `torch.Tensor` (capable de basculer sur Apple Silicon MPS ou GPU CUDA).
   - Le moteur d'autodiffusion **`torch.autograd`** : calcul automatique instantané de toutes les dérivées via `loss.backward()`.
   - Les optimiseurs avancés **`torch.optim`** (`SGD` avec momentum, `Adam`).
   - Les modules de réseaux **`torch.nn`** (`nn.Linear`, `nn.Sigmoid`, `nn.BCELoss`).
6. **Maîtriser la boucle d'apprentissage canonique en 5 lignes** de PyTorch.
7. **Prendre notre revanche sur 1969** : résoudre enfin le problème du **XOR** grâce à un mini-réseau multicouches (MLP) moderne codé en 6 lignes de PyTorch !
8. **Benchmark de performance** : mesurer l'accélération matérielle permise par PyTorch sur puce Apple Silicon (Metal Performance Shaders - MPS).

> 💡 **Pédagogie interactive** : Ce notebook contient des **QCM interactifs** et des cellules de vérification automatique `check()` pour valider vos réponses en direct.  
> 👉 `Shift + Entrée` pour exécuter chaque cellule pas à pas.


In [ ]:
# Programme 1 — Initialisation & Outils interactifs (QCM et validation)
import numpy as np
import matplotlib.pyplot as plt
import random as _random_qcm
import ipywidgets as w
from IPython.display import display
import torch
import torch.nn as nn
import torch.optim as optim

# Relances humoristiques adaptées à la descente de gradient et aux neurones modernes
_RELANCES_HUMOUR = [
    "Aïe ! Ton gradient a explosé vers l'infini. Réduis le learning rate !",
    "Oubli de optimizer.zero_grad() ? Les gradients de l'époque précédente s'accumulent !",
    "Attention : tu as sauté par-dessus la cuvette du minimum global !",
    "Erreur de dimension entre les tenseurs PyTorch et les matrices NumPy !",
    "La fonction de coût remonte au lieu de descendre... As-tu mis un signe '+' au lieu de '-' ?",
    "Le tenseur est resté sur CPU alors que le modèle t'attend sur Apple MPS !"
]

def qcm(question, options, correct, explication=""):
    """
    Affiche un QCM interactif avec ipywidgets.
    correct : index (int) ou liste d'index de la/des bonne(s) réponse(s).
    """
    if isinstance(correct, int):
        correct = [correct]

    boutons = w.RadioButtons(options=options, layout=w.Layout(width='100%'))
    sortie  = w.Output()
    titre   = w.HTML(f"<b>❓ {question}</b>")

    def verifier(change):
        idx = options.index(change['new'])
        sortie.clear_output()
        with sortie:
            if idx in correct:
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#d4edda;color:#155724;border-left:5px solid #28a745'>"
                    f"✅ <b>Bravo, c'est exact !</b><br>{explication}</div>"
                ))
            else:
                humour = _random_qcm.choice(_RELANCES_HUMOUR)
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#f8d7da;color:#721c24;border-left:5px solid #dc3545'>"
                    f"❌ <b>Pas tout à fait...</b> {humour}<br><br>"
                    f"<b>Explication :</b> {explication}</div>"
                ))

    boutons.observe(verifier, names='value')
    display(w.VBox([titre, boutons, sortie]))

def check(nom, valeur, attendu, tol=1e-4):
    """
    Vérifie la valeur d'une variable, d'un array NumPy ou d'un Tensor PyTorch.
    """
    ok = False
    try:
        # Conversion si Tensor PyTorch
        if isinstance(valeur, torch.Tensor):
            valeur = valeur.detach().cpu().numpy()
        if isinstance(attendu, torch.Tensor):
            attendu = attendu.detach().cpu().numpy()

        if isinstance(attendu, np.ndarray) and isinstance(valeur, np.ndarray):
            ok = valeur.shape == attendu.shape and np.allclose(valeur, attendu, atol=tol)
        elif isinstance(attendu, (int, float, np.integer, np.floating)) and isinstance(valeur, (int, float, np.integer, np.floating)):
            ok = abs(float(valeur) - float(attendu)) <= tol
        elif isinstance(attendu, (tuple, list)) and isinstance(valeur, (tuple, list)):
            ok = len(valeur) == len(attendu) and all(abs(float(v) - float(a)) <= tol for v, a in zip(valeur, attendu))
        else:
            ok = bool(valeur == attendu)
    except Exception:
        ok = False

    if ok:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#d4edda;color:#155724;border-left:4px solid #28a745'>"
            f"✅ <b>{nom}</b> est correct ! (Valeur : <code>{repr(valeur)}</code>)</div>"
        ))
    else:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#f8d7da;color:#721c24;border-left:4px solid #dc3545'>"
            f"❌ <b>{nom}</b> n'est pas encore la valeur attendue.<br>"
            f"Ta valeur : <code>{repr(valeur)}</code><br>"
            f"Valeur attendue : <code>{repr(attendu)}</code></div>"
        ))

# Détection de l'accélérateur matériel disponible
mps_disponible = torch.backends.mps.is_available()
cuda_disponible = torch.cuda.is_available()
device_defaut = "mps" if mps_disponible else ("cuda" if cuda_disponible else "cpu")

print(f"✅ Environnement initialisé avec NumPy {np.__version__} et PyTorch {torch.__version__}")
print(f"🚀 Accélération matérielle détectée : '{device_defaut.upper()}' (Apple Metal MPS : {mps_disponible})")
print("✅ Fonctions qcm() et check() prêtes à l'emploi.")


---
## 1 · La Rupture : Pourquoi le Perceptron Bloque (Heaviside vs Sigmoïde)

Dans le Perceptron de Rosenblatt (Jour 3), le neurone calculait la somme pondérée $z = \mathbf{w} \cdot \mathbf{x} + b$, puis appliquait la fonction seuil d'**Heaviside** :
$$\hat{y} = f(z) = \begin{cases} 1 & \text{si } z \ge 0 \\ 0 & \text{si } z < 0 \end{cases}$$

### 🛑 Le drame du gradient nul
Si l'on cherche la dérivée de la fonction d'Heaviside par rapport à $z$ :
- Pour $z > 0$, $f(z) = 1$ est constant, donc **$f'(z) = 0$**.
- Pour $z < 0$, $f(z) = 0$ est constant, donc **$f'(z) = 0$**.
- En $z = 0$, la fonction présente un saut discontinu vertical : la dérivée n'est **pas définie** (ou infinie au sens des distributions de Dirac).

> [!CAUTION]
> **Conséquence dramatique pour l'optimisation** :  
> Une dérivée nulle ($f'(z) = 0$) signifie que la pente locale est **plate comme une table**.  
> Si vous modifiez un poids $w$ d'un petit montant $\Delta w$, la somme $z$ bouge un peu, mais la prédiction $\hat{y}$ **ne bouge pas d'un iota** ! L'erreur reste rigoureusement identique.  
> Mathématiquement, le gradient $\frac{\partial L}{\partial w} = 0$ : l'algorithme ne sait absolument pas s'il doit augmenter ou diminuer $w$ pour réduire l'erreur. L'optimisation par gradient est totalement bloquée.

### 💡 Exemple Numérique : Interrupteur ON/OFF vs Variateur Continu
Imaginons un échantillon où le label attendu est **$y = 1$**, et le neurone calcule actuellement **$z = -0.5$** (l'échantillon est mal classé).
Supposons que nous testions un petit ajustement de poids qui augmente $z$ de $+0.1$ (faisant passer $z$ de $-0.5$ à $-0.4$) :

| Modèle | Avant ($z = -0.5$) | Après ($z = -0.4$) | Variation de sortie $\Delta \hat{y}$ | Pente / Gradient local | Information pour l'apprentissage |
|---|---|---|---|---|---|
| **Heaviside (Rosenblatt)** | $\hat{y} = 0$ | $\hat{y} = 0$ | **$0.0$** | **$\mathbf{0.0}$ (Pente nulle)** | ❌ **Aveugle** : rien n'a bougé, impossible de savoir si on va dans le bon sens ! |
| **Sigmoïde (Moderne)** | $\hat{y} \approx 0.378$ | $\hat{y} \approx 0.401$ | **$+0.023$** | **$+0.235 > 0$** | ✅ **Informatif** : la probabilité augmente vers $1$, le gradient confirme la bonne direction ! |

### 🌊 La Solution : La Fonction Sigmoïde (Neurone Lisse)
Pour rendre le neurone différentiable, on remplace le seuil brutal par une courbe continue en S : la **fonction Sigmoïde** (notée $\sigma$) :

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

![Comparaison Heaviside vs Sigmoïde](pictures/activation_heaviside_vs_sigmoid.png)

### ✨ Les 3 super-pouvoirs de la Sigmoïde :
1. **Différentiable partout** : sa courbe est parfaitement lisse et sa dérivée s'exprime très simplement à partir de sa propre valeur :
   $$\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$$
2. **Interprétable comme une probabilité** : la sortie $\hat{y} = \sigma(z)$ est toujours strictement comprise dans l'intervalle $]0, 1[$.  
   *Exemple : $\hat{y} = 0.92$ signifie « le modèle estime à 92 % la probabilité que cet appareil soit allumé ».*
3. **Pente informative** : plus l'erreur est grande, plus le gradient fournit une correction vigoureuse pour guider les poids dans la bonne direction.


In [ ]:
# Programme 2 — QCM 1 : Différentiabilité et Gradient
qcm(
    "Pourquoi la fonction d'activation d'Heaviside (0 ou 1) empêche-t-elle d'utiliser la descente de gradient ?",
    [
        "Parce que l'exponentielle dans Heaviside produit des valeurs trop grandes (dépassement mémoire float64).",
        "Parce que sa dérivée est nulle presque partout, ce qui empêche le gradient d'indiquer dans quelle direction ajuster les poids.",
        "Parce qu'Heaviside ne fonctionne qu'avec des matrices carrées de dimension 2x2.",
        "Parce qu'elle force le learning rate à rester négatif."
    ],
    correct=1,
    explication="Une dérivée nulle (f'(z) = 0) signifie que la pente est plate. Sans pente, l'algorithme ne sait pas s'il doit augmenter ou diminuer w pour réduire l'erreur !"
)


---
## 2 · La Fonction de Coût (Loss Function) & La Géométrie de l'Optimisation

Dans le Perceptron, on comptait simplement le nombre d'erreurs (un entier discontinu).  
Pour optimiser avec le calcul différentiel, nous avons besoin d'une **fonction de coût continue et différentiable** (notée $L$ ou $J$), qui mesure la gravité des erreurs commises sur l'ensemble des $m$ observations.

### 📉 La Perte Logarithmique (Binary Cross-Entropy / Log-Loss)
Pour une classification binaire avec probabilité prédite $\hat{y}_i = \sigma(\mathbf{w} \cdot \mathbf{x}_i + b)$ et label réel $y_i \in \{0, 1\}$, la fonction de coût standard est la **Binary Cross-Entropy (BCE)** :

$$L(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^m \Big[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \Big]$$

- Si le label réel est **$y_i = 1$** : la perte vaut $-\ln(\hat{y}_i)$. Si $\hat{y}_i \to 1$, la perte tend vers $0$ (parfait). Si $\hat{y}_i \to 0$, la perte explose vers $+\infty$ (punition maximale) !
- Si le label réel est **$y_i = 0$** : la perte vaut $-\ln(1 - \hat{y}_i)$. Si $\hat{y}_i \to 0$, la perte tend vers $0$. Si $\hat{y}_i \to 1$, la perte explose vers $+\infty$ !

### ⛰️ L'Intuition Physique : La Montagne dans le Brouillard
Imaginez que vous êtes au sommet d'une colline brumeuse sans visibilité. Comment rejoindre le village situé au fond de la vallée ?
- Vous palpez le sol sous vos pieds pour sentir **la direction où la pente monte le plus fort** : c'est le **vecteur gradient** $\nabla_{\mathbf{w}} L$.
- Pour descendre vers le minimum, vous faites donc un pas dans la direction **strictement opposée au gradient** : $-\nabla_{\mathbf{w}} L$.

$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta \cdot \nabla_{\mathbf{w}} L$$
$$b_{t+1} = b_t - \eta \cdot \frac{\partial L}{\partial b}$$

où **$\eta$** (*learning rate*, taux d'apprentissage) contrôle la taille du pas à chaque itération.

![Géométrie de la Descente de Gradient](pictures/gradient_descent_bowl.png)

### ⚠️ Le Rôle Critique du Taux d'Apprentissage $\eta$ :
- **$\eta$ trop petit (ex: $10^{-6}$)** : la descente avance à la vitesse d'un escargot, nécessitant des millions d'itérations.
- **$\eta$ trop grand (ex: $10.0$)** : l'algorithme fait des pas géants, saute par-dessus la cuvette, oscille violemment et **diverge vers l'infini**.
- **$\eta$ optimal (ex: $0.1$ à $0.5$)** : convergence rapide, fluide et stable vers le minimum global.


In [ ]:
# Programme 3 — QCM 2 : Direction du Gradient et Pas d'Apprentissage
qcm(
    "Dans l'algorithme de descente de gradient w = w - η * ∇L, pourquoi y a-t-il un signe moins devant le gradient ?",
    [
        "Parce que le learning rate η est toujours un nombre négatif.",
        "Parce que le gradient pointe dans la direction de la plus forte CROISSANCE de la fonction de coût, donc son opposé (-∇L) pointe vers la plus forte DÉCROISSANCE.",
        "Pour éviter que la matrice des poids ne devienne singulière.",
        "C'est une convention arbitraire héritée de Scikit-Learn sans signification géométrique."
    ],
    correct=1,
    explication="Par définition mathématique, le gradient ∇L est le vecteur qui pointe vers là où la pente grimpe le plus vite. Pour minimiser l'erreur (descendre dans la vallée), il faut impérativement aller dans le sens inverse : -∇L !"
)


---
## 3 · Descente de Gradient "From Scratch" en Pur NumPy

Avant de déléguer les calculs aux modules modernes, maîtrisons les équations fondamentales en les codant nous-mêmes en NumPy pur.

Pour un jeu de données de $m$ observations $\mathbf{X} \in \mathbb{R}^{m \times p}$ et labels $\mathbf{y} \in \{0, 1\}^m$ :
1. **Combinaison linéaire** : $\mathbf{z} = \mathbf{X} \mathbf{w} + b$
2. **Activation sigmoïde** : $\mathbf{\hat{y}} = \sigma(\mathbf{z}) = \frac{1}{1 + e^{-\mathbf{z}}}$
3. **Fonction de coût BCE** : $L(\mathbf{w}, b) = -\frac{1}{m} \sum [y \ln(\hat{y}) + (1-y)\ln(1-\hat{y})]$

### 📐 La Règle de Chaîne (*Chain Rule*) : La Magie de la Dérivation
En dérivant la perte $L$ par rapport aux poids $\mathbf{w}$ en utilisant la règle de dérivation en chaîne :
$$\frac{\partial L}{\partial \mathbf{w}} = \frac{\partial L}{\partial \mathbf{\hat{y}}} \cdot \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} \cdot \frac{\partial \mathbf{z}}{\partial \mathbf{w}}$$

Toutes les simplifications algébriques mènent à un résultat d'une élégance absolue :
$$\nabla_{\mathbf{w}} L = \frac{1}{m} \mathbf{X}^T (\mathbf{\hat{y}} - \mathbf{y})$$
$$\frac{\partial L}{\partial b} = \frac{1}{m} \sum_{i=1}^m (\hat{y}_i - y_i)$$

> [!NOTE]
> **Remarquez la filiation avec Rosenblatt !**  
> La formule du gradient repose sur le terme d'erreur $(\mathbf{\hat{y}} - \mathbf{y})$, tout comme Rosenblatt utilisait $(y_i - \hat{y}_i)$.  
> Mais ici, l'erreur est **continue**, pondérée par les probabilités, et calculée **sur l'ensemble du lot de données** d'un seul coup grâce au produit matriciel `X.T @ error` !


### ✏️ Exercice 1 — Sigmoïde et Perte BCE en NumPy

Implémentons les deux fonctions élémentaires de notre neurone différentiable :
1. `sigmoid(z)` : calcule l'activation $\sigma(z) = \frac{1}{1 + e^{-z}}$.
2. `bce_loss(y_true, y_pred)` : calcule la perte moyenne d'entropie croisée binaire :
   $$L = -\frac{1}{m} \sum_{i=1}^m \Big[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \Big]$$

---

### 💡 Décorticage pas-à-pas : Comment calculer la somme dans `bce_loss` ?

La formule mathématique peut sembler impressionnante avec son symbole $\sum$, mais en NumPy, elle s'écrit en **une seule ligne très simple** ! Voici le mécanisme :

#### Étape 1 : Le calcul élément par élément (Vectorisation)
Soient `y_true` et `y_pred` deux tableaux NumPy de taille $m$ (les labels réels et les probabilités prédites) :
- `np.log(y_pred_clipped)` calcule le logarithme népérien pour chaque exemple.
- L'expression :
  ```python
  termes = y_true * np.log(y_pred_clipped) + (1.0 - y_true) * np.log(1.0 - y_pred_clipped)
  ```
  s'évalue **en une seule fois pour tous les exemples** sans aucune boucle `for` !
  Le résultat `termes` est un vecteur 1D de longueur $m$, où chaque élément $i$ contient la valeur $\big[ y_i \ln(\hat{y}_i) + (1 - y_i) \ln(1 - \hat{y}_i) \big]$.

#### Étape 2 : Comment faire la somme $\sum$ divisée par $m$ ?
Que signifie mathématiquement l'opération $\frac{1}{m} \sum_{i=1}^m (\dots)$ ?  
Additionner $m$ valeurs puis diviser le total par $m$, **c'est la définition exacte de la moyenne** !

En NumPy, vous avez donc deux options rigoureusement identiques :
- **Option A (Recommandée & la plus propre)** : utiliser `np.mean()` !
  ```python
  # np.mean calcule automatiquement la somme de tous les éléments divisée par m
  perte = -np.mean(termes)
  ```
- **Option B (Explicite avec la somme)** : utiliser `np.sum()` divisé par $m$ :
  ```python
  m = len(y_true)
  perte = -(1.0 / m) * np.sum(termes)
  ```

#### Étape 3 : Pourquoi le signe moins `'-'` devant ?
Comme les probabilités $\hat{y}$ sont comprises entre 0 et 1, leur logarithme $\ln(\hat{y})$ est **toujours négatif** (par exemple $\ln(0.9) \approx -0.105$, $\ln(0.1) \approx -2.302$).  
La moyenne calculée à l'étape 2 est donc un nombre négatif.  
Le signe `'-'` devant permet de transformer ce résultat en un coût **positif** ($L \ge 0$) :  
- Si le modèle prédit juste ($\hat{y} \approx y$), la perte est très proche de $0$.
- Plus le modèle se trompe lourdement, plus la perte devient un grand nombre positif.

---

### 🛡️ Qu'est-ce que `np.clip` et pourquoi est-il indispensable ici ?

La fonction **`np.clip(a, a_min, a_max)`** de NumPy sert à **écrêter** (borner) toutes les valeurs d'un tableau à l'intérieur d'un intervalle défini $[a_{\min}, a_{\max}]$ :
- Toute valeur strictement inférieure à $a_{\min}$ est ramenée à $a_{\min}$.
- Toute valeur strictement supérieure à $a_{\max}$ est ramenée à $a_{\max}$.
- Les valeurs déjà situées entre les deux bornes restent strictement inchangées.

```python
# Exemple intuitif d'écrêtage :
valeurs = np.array([-15.0, 3.0, 7.5, 42.0])
np.clip(valeurs, a_min=0.0, a_max=10.0)
# Résultat : array([ 0.0,  3.0,  7.5, 10.0])
```

#### 🚨 Pourquoi l'utilise-t-on impérativement en Machine Learning (Stabilité Numérique) ?
En calcul sur ordinateur, les nombres flottants 64 bits (`float64`) ont des limites physiques de représentation (norme IEEE 754) :

1. **Éviter le crash par *Overflow* dans `sigmoid(z)` avec `np.clip(z, -500, 500)`** :
   - Si au cours de la descente de gradient, $z$ devient très négatif (ex: $z = -1000$), le calcul $e^{-(-1000)} = e^{1000}$ dépasse le plus grand nombre flottant possible ($\approx 1.8 \times 10^{308}$).
   - NumPy déclenche alors une erreur d'*overflow* et renvoie `inf`.
   - En appliquant `z_clipped = np.clip(z, -500, 500)`, $z$ est borné à $-500$. Or $e^{500} \approx 1.4 \times 10^{217}$ se calcule sans encombre et donne $\sigma(-500) \approx 0.0$, ce qui est la valeur asymptotique exacte de la sigmoïde !

2. **Éviter le crash `ln(0) = -inf` et les `NaN` dans `bce_loss` avec `np.clip(y_pred, 1e-15, 1.0 - 1e-15)`** :
   - Dans la formule de la perte BCE, on calcule $\ln(\hat{y})$ et $\ln(1 - \hat{y})$.
   - Si le modèle prédit une probabilité extrême $\hat{y} = 0.0$ ou $\hat{y} = 1.0$, on tente d'évaluer $\ln(0)$, qui vaut mathématiquement $-\infty$.
   - Pire encore : $0 \times (-\infty)$ produit un **`NaN`** (*Not a Number*). Un seul `NaN` contamine ensuite l'intégralité des calculs de moyennes, des gradients et des poids du réseau !
   - En écrivant `y_pred_clipped = np.clip(y_pred, 1e-15, 1.0 - 1e-15)`, la probabilité prédite reste bornée dans $[10^{-15}, 1 - 10^{-15}]$. Comme $\ln(10^{-15}) \approx -34.5$, la punition pour une mauvaise prédiction est très sévère, mais le résultat reste un nombre fini sans jamais corrompre l'apprentissage avec un `NaN` !


In [ ]:
# Programme 4 — Exercice 1 : Fonctions Sigmoïde et BCE en NumPy
def sigmoid(z):
    """
    Calcule l'activation sigmoïde 1 / (1 + exp(-z)).
    """
    # Remplacez les '...' par votre formule vectorisée :
    z_clipped = np.clip(z, -500, 500)
    return ...

def bce_loss(y_true, y_pred, eps=1e-15):
    """
    Calcule la perte moyenne d'entropie croisée binaire (BCE).
    """
    # Remplacez les '...' par votre formule vectorisée :
    # Astuce : la somme divisée par m correspond directement à -np.mean(...) !
    y_pred_clipped = np.clip(y_pred, eps, 1.0 - eps)
    perte = ...
    return perte


In [ ]:
# Programme 5 — Test Exercice 1
if "np" not in globals() or "check" not in globals():
    raise NameError("⚠️ Pensez à exécuter la cellule 'Programme 1' (tout en haut) pour initialiser NumPy et les outils interactifs !")

_z_test = np.array([-2.0, 0.0, 2.0])
_sig_ref = 1.0 / (1.0 + np.exp(-_z_test))
_sig_eleve = sigmoid(_z_test)

check("sigmoid([-2, 0, 2])", _sig_eleve, _sig_ref)

_y_reel = np.array([0.0, 1.0, 1.0])
_y_pred = np.array([0.1, 0.9, 0.8])
_loss_ref = -np.mean(_y_reel * np.log(_y_pred) + (1.0 - _y_reel) * np.log(1.0 - _y_pred))
_loss_eleve = bce_loss(_y_reel, _y_pred)

check("bce_loss", _loss_eleve, _loss_ref)


### ✏️ Exercice 2 — Le Calcul Vectorisé du Gradient en NumPy

Écrivons la fonction `compute_gradients(X, y_true, y_pred)` chargée de calculer $\nabla_{\mathbf{w}} L$ et $\frac{\partial L}{\partial b}$ :
- Soit $m = \mathbf{X}.\text{shape}[0]$ le nombre d'observations.
- Le vecteur d'erreur résiduelle est : `error = y_pred - y_true` (de forme `(m,)`).
- Le gradient des poids est le produit matriciel de la transposée de $\mathbf{X}$ avec l'erreur :
  $$\mathbf{dw} = \frac{1}{m} \mathbf{X}^T \mathbf{error}$$
- Le gradient du biais est la moyenne des erreurs :
  $$db = \frac{1}{m} \sum_{i=1}^m error_i = \text{np.mean}(error)$$


In [ ]:
# Programme 6 — Exercice 2 : Calcul des Gradients Vectorisés
def compute_gradients(X, y_true, y_pred):
    """
    Calcule le gradient de la perte BCE par rapport à w et b.
    Retourne : (dw, db)
    """
    m = X.shape[0]
    error = y_pred - y_true
    
    # Remplacez les '...' par vos calculs vectorisés :
    dw = ...
    db = ...
    
    return dw, db


In [ ]:
# Programme 7 — Test Exercice 2
_X_demo = np.array([[1.0, 2.0], [2.0, 1.0], [0.5, 1.5]])
_y_demo = np.array([1.0, 0.0, 1.0])
_y_p_demo = np.array([0.8, 0.4, 0.7])

_m_demo = _X_demo.shape[0]
_err_demo = _y_p_demo - _y_demo
_dw_ref = (1.0 / _m_demo) * (_X_demo.T @ _err_demo)
_db_ref = np.mean(_err_demo)

_dw_eleve, _db_eleve = compute_gradients(_X_demo, _y_demo, _y_p_demo)

check("dw (gradient des poids)", _dw_eleve, _dw_ref)
check("db (gradient du biais)", _db_eleve, _db_ref)


### ✏️ Exercice 3 — La Classe Complète `NeuroneGradientNumPy`

Assemblons ces briques dans un modèle orienté objet complet :
- `predict_proba(self, X)` : calcule $\sigma(\mathbf{X} \mathbf{w} + b)$.
- `predict(self, X, seuil=0.5)` : retourne les prédictions binaires `np.where(proba >= seuil, 1, 0)`.
- `fit(self, X, y)` :
  1. Initialise $\mathbf{w}$ aléatoirement avec de petites valeurs et $b = 0.0$.
  2. Pour chaque époque dans `range(self.n_epochs)` :
     - Calcule les probabilités `y_pred = self.predict_proba(X)`.
     - Calcule la perte BCE et l'enregistre dans `self.loss_history_`.
     - Calcule les gradients `dw, db = compute_gradients(X, y, y_pred)`.
     - Met à jour les paramètres par descente de gradient :
       $$\mathbf{w} \leftarrow \mathbf{w} - \eta \cdot \mathbf{dw}$$
       $$b \leftarrow b - \eta \cdot db$$


In [ ]:
# Programme 8 — Exercice 3 : Classe NeuroneGradientNumPy
class NeuroneGradientNumPy:
    def __init__(self, lr=0.5, n_epochs=100, random_state=42):
        self.lr = lr
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.w = None
        self.b = 0.0
        self.loss_history_ = []

    def predict_proba(self, X):
        """Calcule les probabilités prédites p(y=1|x) = sigmoid(X @ w + b)."""
        # Remplacez les '...' :
        z = ...
        return sigmoid(z)

    def predict(self, X, seuil=0.5):
        """Convertit les probabilités en classes binaires {0, 1}."""
        return np.where(self.predict_proba(X) >= seuil, 1, 0)

    def fit(self, X, y):
        """Entraîne le neurone par descente de gradient batch."""
        rng = np.random.default_rng(self.random_state)
        self.w = rng.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b = 0.0
        self.loss_history_ = []

        for epoch in range(self.n_epochs):
            # 1. Forward pass (probabilités)
            y_pred = self.predict_proba(X)
            
            # 2. Calcul et sauvegarde de la perte
            loss = bce_loss(y, y_pred)
            self.loss_history_.append(loss)
            
            # 3. Calcul des gradients
            dw, db = compute_gradients(X, y, y_pred)
            
            # 4. Mise à jour des paramètres (Descente de Gradient)
            # Remplacez les '...' :
            self.w -= ...
            self.b -= ...

        return self


In [ ]:
# Programme 9 — Test Exercice 3 & Entraînement sur Blobs Synthétiques
# Générons un jeu de données 2D synthétique (100 observations)
rng_data = np.random.default_rng(seed=42)
X_c0 = rng_data.normal(loc=[2.0, 2.0], scale=0.6, size=(50, 2))
X_c1 = rng_data.normal(loc=[5.0, 5.0], scale=0.6, size=(50, 2))
X_blobs = np.vstack([X_c0, X_c1])
y_blobs = np.array([0.0] * 50 + [1.0] * 50)

# Entraînement de notre modèle NumPy
model_np = NeuroneGradientNumPy(lr=0.5, n_epochs=100, random_state=42)
model_np.fit(X_blobs, y_blobs)

preds_np = model_np.predict(X_blobs)
acc_np = np.mean(preds_np == y_blobs)

check("Accuracy NumPy Blobs", acc_np, 1.0)
check("Décroissance de la perte (dernière perte < première perte)", model_np.loss_history_[-1] < model_np.loss_history_[0], True)
print(f"🎯 Précision finale : {acc_np * 100:.1f} %")
print(f"📉 Perte initiale : {model_np.loss_history_[0]:.4f} ➔ Perte finale : {model_np.loss_history_[-1]:.4f}")


In [ ]:
# Programme 10 — Visualisation Graphique : Convergence et Frontière
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# 1. Courbe de décroissance de la perte BCE
ax1.plot(range(1, len(model_np.loss_history_) + 1), model_np.loss_history_, color='#0969da', lw=2.5)
ax1.set_title("Courbe de Convergence de la Descente de Gradient (NumPy)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Époque d'entraînement", fontsize=11)
ax1.set_ylabel("Perte Log-Loss (BCE)", fontsize=11)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.annotate(f"Perte finale : {model_np.loss_history_[-1]:.4f}", 
             xy=(100, model_np.loss_history_[-1]), xytext=(65, 0.35),
             arrowprops=dict(facecolor='#0969da', shrink=0.08, width=1.5, headwidth=6),
             fontweight='bold', color='#0969da')

# 2. Frontière et gradients de probabilité dans l'espace 2D
x_min, x_max = 0.0, 7.0
y_min, y_max = 0.0, 7.0
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
probs = model_np.predict_proba(grid).reshape(xx.shape)

contour = ax2.contourf(xx, yy, probs, levels=25, cmap='coolwarm', alpha=0.3)
cbar = plt.colorbar(contour, ax=ax2)
cbar.set_label("Probabilité prédite $\hat{y} = p(Classe=1)$", fontsize=10)

# Tracé de la ligne critique p = 0.5 (z = 0)
ax2.contour(xx, yy, probs, levels=[0.5], colors=['#cf222e'], linewidths=[2.5], linestyles=['--'])

ax2.scatter(X_blobs[y_blobs == 0, 0], X_blobs[y_blobs == 0, 1], color='#0969da', s=70, edgecolors='black', label='Classe 0')
ax2.scatter(X_blobs[y_blobs == 1, 0], X_blobs[y_blobs == 1, 1], color='#cf222e', s=70, marker='^', edgecolors='black', label='Classe 1')
ax2.set_title("Espace 2D : Probabilités Lisses & Frontière z = 0", fontsize=12, fontweight='bold')
ax2.set_xlabel("$x_1$", fontsize=11)
ax2.set_ylabel("$x_2$", fontsize=11)
ax2.legend(loc='upper left')
ax2.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()


### 🎬 En direct : L'Animation Complète de la Descente de Gradient

Observons concrètement la dynamique conjointe de l'optimisation à chaque époque :

<p align="center">
  <img src="pictures/descente_gradient_convergence.gif" alt="Animation de la convergence de la descente de gradient" width="95%" style="border-radius: 8px; box-shadow: 0 4px 16px rgba(0,0,0,0.3);" />
</p>

> [!TIP]
> **Ce que montre l'animation synchronisée (61 époques) :**
> - **À gauche (Espace 2D des données)** : Les probabilités continues $\hat{y} = \sigma(z)$ s'ajustent en temps réel (gradient du bleu $\approx 0$ au rouge $\approx 1$). La droite frontière rouge $p = 0.5$ ($z = 0$) démarre à une position arbitraire, puis pivote et translate pour s'installer parfaitement au centre du couloir séparateur (**100 % de précision**).
> - **En haut à droite (Cuvette de perte & Trajectoire des poids)** : Dans le plan $(w_1, w_2)$, la bille rouge démarre près de l'origine (point bleu $w_0$) et glisse le long des lignes de niveau dans la direction du vecteur $-\eta \nabla_{\mathbf{w}} L$ pour atteindre le minimum global optimal $\mathbf{w}^*$ (étoile verte).
> - **En bas à droite (Courbe d'apprentissage Log-Loss)** : La perte BCE s'effondre de façon exponentielle, passant de $0.70$ à $0.13$ au fur et à mesure que les gradients diminuent à l'approche du fond de la cuvette.


---
## 4 · Le Plafond de Verre : Pourquoi NumPy Ne Suffit Plus pour l'IA Moderne

Notre implémentation NumPy fonctionne à merveille pour un neurone unique. Mais dès qu'on cherche à concevoir des architectures plus ambitieuses, **NumPy se heurte à trois barrières infranchissables** :

### 🧱 1. Le Cauchemar de la Dérivation Manuelle
Sur un seul neurone, dériver $\nabla_{\mathbf{w}} L$ a pris quelques lignes de calcul au tableau.  
Mais que se passe-t-il si vous empilez **3 couches, 10 couches ou 96 couches** avec des convolutions et des mécanismes d'attention (comme dans les LLMs ou les réseaux de vision) ?
- La règle de chaîne produit des milliers de produits matriciels en cascade (rétropropagation).
- Chaque modification d'architecture (changer une fonction d'activation, ajouter une couche) imposerait de **recalculer toutes les dérivées à la main sur papier**. C'est humainement impossible !

### ⚡ 2. L'Impossibilité d'Exploiter l'Accélération Matérielle (GPU & Apple Silicon)
NumPy est un outil conçu pour le **CPU**. Il ne sait pas envoyer de matrices sur :
- Les GPU de calcul intensif (**NVIDIA CUDA**).
- Les puces graphiques unifiées des Macs modernes (**Apple Silicon Metal Performance Shaders - MPS**).  
Or, un GPU ou un GPU Apple Silicon exécute les multiplications matricielles sur des milliers de cœurs en parallèle, souvent **10 à 100 fois plus vite qu'un CPU** !

### 🎯 3. Des Algorithmes d'Optimisation Trop Rudimentaires
La descente de gradient classique (SGD simple) souffre sur les paysages de perte réels :
- Elle ralentit sur les plateaux où la pente est faible.
- Elle oscille violemment dans les ravins étroits.  
L'IA moderne a besoin d'optimiseurs avancés comme **Adam (Adaptive Moment Estimation)**, qui calculent dynamiquement un taux d'apprentissage adapté à chaque poids individuel.

---

### 🗺️ Panorama Comparatif : Quel Outil Choisir pour Vos Modèles ?

| Outil | Rôle Principal | Calcul des Gradients | Support Matériel | Cas d'Usage Idéal |
|:---|:---|:---:|:---:|:---|
| **NumPy** <br>*(Le socle)* | Algèbre linéaire et calcul matriciel sur tableaux NumPy (`np.array`) | ❌ **Manuel**<br>(calcul papier obligatoire) | 💻 **CPU uniquement** | • Prototypage et algèbre de base<br>• Prétraitement de features<br>• Comprendre les maths sous le capot |
| **Scikit-Learn** <br>*(Boîte à outils)* | Modèles de ML classiques (Random Forest, SVM, Régression Logistique) | ⚙️ **Interne**<br>(algorithmes clé en main) | 💻 **CPU**<br>(multithreading) | • Données tabulaires classiques<br>• Pipelines industriels standards<br>• Inadapté au Deep Learning custom |
| **PyTorch (`torch`)** <br>*(Standard IA)* | Tenseurs accélérés, réseaux modulaires `torch.nn`, optimiseurs avancés | ✅ **Autograd automatique**<br>(`loss.backward()`) | 🚀 **CPU, GPU & Apple MPS**<br>(accélération x10 à x100) | • **Réseaux de neurones profonds**<br>• Modèles modernes (CNN, Transformers)<br>• Recherche & industrie mondiale |
| **JAX (Google)** <br>*(Alternative)* | Programmation fonctionnelle pure, compilation JIT XLA haute vitesse | ✅ **Différentiation fonctionnelle**<br>(`jax.grad`) | 🚀 **CPU, GPU & Google TPU** | • Recherche avancée en IA & physique<br>• Calculs scientifiques à grande échelle |


In [ ]:
# Programme 11 — QCM 3 : Les Limites de NumPy
qcm(
    "Quelle est la principale innovation apportée par un framework moderne comme PyTorch par rapport à NumPy pour entraîner des réseaux profonds ?",
    [
        "PyTorch remplace Python par du code assembleur écrit manuellement pour chaque ordinateur.",
        "Le moteur de différentiation automatique (Autograd) qui calcule tous les gradients à notre place, combiné au support natif du matériel accélérateur (GPU / Apple MPS).",
        "PyTorch supprime totalement le besoin de définir une fonction de perte.",
        "PyTorch garantit que tous les problèmes deviennent linéairement séparables."
    ],
    correct=1,
    explication="PyTorch résout les deux plus gros goulets d'étranglement de NumPy : il calcule les gradients analytiques automatiquement (plus besoin de dériver à la main !) et exécute les calculs sur GPU/MPS avec des optimiseurs ultra-rapides."
)


---
## 5 · Bienvenue dans l'Ère Moderne : Les Neurones avec PyTorch (`torch`)

Créé par Meta AI et adopté universellement par les laboratoires de recherche et l'industrie (OpenAI, Hugging Face, Tesla...), **PyTorch** est le standard mondial du Deep Learning.

Examinons les 4 piliers de PyTorch :

### 1. `torch.Tensor` : Le Tenseur Accéléré
Le `torch.Tensor` est l'équivalent direct du tableau NumPy (`np.array`), mais avec deux capacités majeures en plus :
> *(Note de vocabulaire : le type interne d'un tableau NumPy s'appelle formellement `np.ndarray` pour « N-Dimensional Array », mais c'est exactement l'objet que vous créez avec `np.array([...])`).*
- **Accélération matérielle** : il peut résider en mémoire GPU ou Apple Silicon : `x = x.to("mps")` ou `x.to("cuda")`.
- **Mémoire de calcul** : il peut mémoriser son historique d'opérations pour calculer son gradient : `requires_grad=True`.
- La conversion entre NumPy et PyTorch est instantanée et sans copie inutile :
  ```python
  x_torch = torch.from_numpy(x_numpy)
  x_numpy = x_torch.numpy()
  ```

### 2. `torch.autograd` : La Différentiation Automatique
Pendant la propagation avant (*Forward pass*), PyTorch enregistre en direct le **graphe de calcul dynamique** reliant les entrées, les poids et la fonction de perte.  
Dès que vous appelez **`loss.backward()`**, le moteur remonte tout le graphe en sens inverse et dépose la dérivée exacte $\frac{\partial L}{\partial w}$ directement dans l'attribut `w.grad` !

#### 🔄 Le Graphe de Calcul PyTorch : Du Forward au Backward

| Étape du Graphe | Sens du Flux | Opération PyTorch | Rôle Mathématique & Données |
|:---|:---:|:---|:---|
| **1. Entrées & Paramètres** | ➡️ Départ | `X` (données), `W, b` | `requires_grad=True` active le suivi des gradients sur les poids $W$ et le biais $b$. |
| **2. Combinaison linéaire** | ➡️ *Forward* | `z = layer(X)` (`nn.Linear`) | Calcule la somme pondérée $z = \mathbf{X}\mathbf{W} + b$. |
| **3. Activation** | ➡️ *Forward* | `y_pred = sigmoid(z)` (`nn.Sigmoid`) | Calcule les probabilités continues $\hat{y} = \sigma(z) \in ]0, 1[$. |
| **4. Fonction de Coût** | 🎯 Cible | `loss = criterion(y_pred, y)` | Évalue l'erreur globale sous la forme d'un **scalaire unique** $L$ (perte BCE). |
| **5. Rétropropagation** | ⬅️ **Backward** | **`loss.backward()`** | **Magie d'Autograd** : remonte la chaîne en sens inverse et dépose $\frac{\partial L}{\partial W}$ dans `W.grad` et $\frac{\partial L}{\partial b}$ dans `b.grad` ! |
| **6. Mise à jour des Poids** | ⚡ Optimiseur | `optimizer.step()` | Ajuste les poids : $W \leftarrow W - \eta \cdot W.\text{grad}$ (ou via Adam). |

> [!NOTE]
> **Pourquoi est-ce une rupture fondamentale par rapport à NumPy ?**  
> - En **NumPy**, vous deviez dériver à la main au tableau et coder manuellement la formule `dw = (1/m) * X.T @ (y_pred - y)`.  
> - En **PyTorch**, vous ne dérivez plus jamais rien : un seul appel à `loss.backward()` calcule automatiquement les dérivées partielles exactes par la règle de chaîne, que le réseau comporte 1 neurone ou 100 couches !

### 3. `torch.nn` : Les Couches et Fonctions de Perte Prêtes à l'Emploi
Plus besoin de coder $X @ w + b$ ni la sigmoïde :
- `nn.Linear(in_features=2, out_features=1)` : combinaison linéaire complète avec initialisation optimale des poids.
- `nn.Sigmoid()` : activation sigmoïde vectorisée.
- `nn.BCELoss()` : perte d'entropie croisée binaire exacte.

### 4. `torch.optim` : Les Optimiseurs Modernes
Fini la mise à jour manuelle `w -= lr * dw` :
- `optim.SGD(model.parameters(), lr=0.1, momentum=0.9)` : descente de gradient avec impulsion pour franchir les plateaux.
- `optim.Adam(model.parameters(), lr=0.05)` : l'optimiseur roi, qui adapte le pas d'apprentissage à chaque coordonnée.

---

### 🌟 Le Pattern d'Or : La Boucle Canonique PyTorch en 5 Lignes
Quelle que soit la taille du modèle (un neurone unique ou un réseau de 70 milliards de paramètres), la boucle d'apprentissage suit **toujours ces 5 étapes immuables** :

```python
for epoch in range(n_epochs):
    optimizer.zero_grad()               # 1. Remettre à zéro les gradients de l'époque précédente
    predictions = model(X_tensor)       # 2. FORWARD : calculer les prédictions
    loss = criterion(predictions, y)    # 3. LOSS : évaluer l'erreur scalaire
    loss.backward()                     # 4. BACKWARD (Autograd) : propager les dérivées
    optimizer.step()                    # 5. STEP (Optimiseur) : ajuster les poids w et b
```


### ✏️ Exercice 4 — Entraîner le Neurone Moderne en PyTorch

Construisons et entraînons notre premier neurone moderne avec PyTorch sur le jeu de données `X_blobs` :
1. Convertissez `X_blobs` et `y_blobs` en tenseurs PyTorch de type `torch.float32`.  
   *(Attention : pour `y`, PyTorch attend une forme `(N, 1)`, utilisez `.unsqueeze(1)` ou `.reshape(-1, 1)`).*
2. Créez le modèle en combinant une couche linéaire et une sigmoïde :
   ```python
   model_torch = nn.Sequential(
       nn.Linear(in_features=2, out_features=1),
       nn.Sigmoid()
   )
   ```
3. Définissez la perte `criterion = nn.BCELoss()` et l'optimiseur `optimizer = optim.Adam(model_torch.parameters(), lr=0.1)`.
4. Remplissez les 5 étapes de la boucle d'entraînement PyTorch.


In [ ]:
# Programme 12 — Exercice 4 : Entraînement du Neurone Moderne avec PyTorch
# 1. Conversion des données NumPy en Tenseurs PyTorch
X_tensor = torch.tensor(X_blobs, dtype=torch.float32)
y_tensor = torch.tensor(y_blobs, dtype=torch.float32).unsqueeze(1)

# 2. Définition du modèle avec torch.nn
torch.manual_seed(42)
model_torch = nn.Sequential(
    nn.Linear(2, 1),
    nn.Sigmoid()
)

# 3. Fonction de perte et Optimiseur
criterion = nn.BCELoss()
optimizer = optim.Adam(model_torch.parameters(), lr=0.1)

# 4. Boucle d'entraînement canonique PyTorch (100 époques)
loss_history_torch = []

for epoch in range(100):
    # Étape 1 : Réinitialiser les gradients à zéro
    # Remplacez les '...' par votre appel :
    ...
    
    # Étape 2 : Propagation avant (Forward pass)
    predictions = ...
    
    # Étape 3 : Calcul de la perte BCE
    loss = ...
    loss_history_torch.append(loss.item())
    
    # Étape 4 : Rétropropagation automatique des gradients (Autograd)
    ...
    
    # Étape 5 : Mise à jour des poids par l'optimiseur Adam
    ...

print("✅ Entraînement PyTorch terminé !")


In [ ]:
# Programme 13 — Test Exercice 4 & Évaluation
with torch.no_grad():
    # Prédictions du modèle PyTorch (seuil à 0.5)
    preds_proba_torch = model_torch(X_tensor).numpy()
    preds_binary_torch = (preds_proba_torch >= 0.5).astype(int).squeeze()

acc_torch = np.mean(preds_binary_torch == y_blobs)

check("Accuracy PyTorch Blobs", acc_torch, 1.0)
check("Décroissance de la perte PyTorch", loss_history_torch[-1] < loss_history_torch[0], True)
check("Perte finale inférieure à 0.15", loss_history_torch[-1] < 0.15, True)

print(f"🎯 Précision PyTorch : {acc_torch * 100:.1f} %")
print(f"📉 Perte initiale : {loss_history_torch[0]:.4f} ➔ Perte finale : {loss_history_torch[-1]:.4f}")
print(f"⚖️ Poids appris (W) : {model_torch[0].weight.detach().numpy().round(3)}")
print(f"🎯 Biais appris (b) : {model_torch[0].bias.detach().numpy().round(3)}")


In [ ]:
# Programme 14 — Comparaison Visuelle : NumPy (SGD) vs PyTorch (Adam)
plt.figure(figsize=(10, 5))
plt.plot(range(1, 101), model_np.loss_history_, label="NumPy (Descente de Gradient standard, η=0.5)", color="#cf222e", lw=2)
plt.plot(range(1, 101), loss_history_torch, label="PyTorch (Optimiseur Adam, lr=0.1)", color="#0969da", lw=2.5)

plt.title("NumPy (SGD pur) vs PyTorch (Adam) : Vitesse de Convergence", fontsize=13, fontweight='bold')
plt.xlabel("Époque", fontsize=11)
plt.ylabel("Perte BCE", fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11, frameon=True, facecolor="white", edgecolor="#d0d7de")
plt.tight_layout()
plt.show()


---
## 6 · La Revanche sur 1969 : Résoudre le XOR avec un MLP PyTorch

Au Jour 3, nous avons vu que le Perceptron de Rosenblatt était **mathématiquement incapable d'apprendre la fonction XOR** (0 erreur impossible car le problème n'est pas linéairement séparable). Cela avait causé la mise au ban des réseaux de neurones pendant plus d'une décennie.

Grâce à la différentiabilité et à PyTorch, regardez à quel point il est devenu enfantin de combiner plusieurs neurones en couches successives (**Multi-Layer Perceptron / MLP**) :
- **Couche d'entrée** : 2 caractéristiques ($x_1, x_2$).
- **Couche cachée** : 4 neurones avec fonction d'activation non-linéaire **ReLU** ($f(x) = \max(0, x)$).
- **Couche de sortie** : 1 neurone avec activation **Sigmoïde** pour la probabilité finale.

```python
mlp_xor = nn.Sequential(
    nn.Linear(2, 4),   # Couche cachée : 4 neurones modernes
    nn.ReLU(),         # Non-linéarité capable de tordre l'espace géométrique
    nn.Linear(4, 1),   # Couche de sortie
    nn.Sigmoid()       # Probabilité finale
)
```

Testons-le immédiatement !


In [ ]:
# Programme 15 — Entraînement du MLP PyTorch sur la Porte XOR
# Données de la porte XOR (non linéairement séparable)
X_xor = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]], dtype=torch.float32)
y_xor = torch.tensor([[0.], [1.], [1.], [0.]], dtype=torch.float32)

torch.manual_seed(42)
mlp_xor = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)

opt_xor = optim.Adam(mlp_xor.parameters(), lr=0.05)
crit_xor = nn.BCELoss()

# Boucle d'apprentissage du MLP (500 époques)
for epoch in range(500):
    opt_xor.zero_grad()
    preds = mlp_xor(X_xor)
    loss_x = crit_xor(preds, y_xor)
    loss_x.backward()
    opt_xor.step()

# Évaluation sur le XOR
with torch.no_grad():
    preds_xor_bin = (mlp_xor(X_xor) >= 0.5).int().squeeze().numpy()

acc_xor = np.mean(preds_xor_bin == y_xor.squeeze().numpy())

print("🎯 RÉSULTATS DU MLP PYTORCH SUR LA PORTE XOR :")
print("  Entrées : [[0,0], [0,1], [1,0], [1,1]]")
print("  Prédictions :", preds_xor_bin)
print("  Attendu     :", [0, 1, 1, 0])
print(f"  Précision finale : {acc_xor * 100:.1f} % (Perte : {loss_x.item():.4f})")

# Visualisation de la frontière non-linéaire qui résout le XOR
fig, ax = plt.subplots(figsize=(7, 6))
xx_x, yy_x = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid_xor = torch.tensor(np.c_[xx_x.ravel(), yy_x.ravel()], dtype=torch.float32)

with torch.no_grad():
    z_grid = mlp_xor(grid_xor).reshape(xx_x.shape).numpy()

ax.contourf(xx_x, yy_x, z_grid, levels=20, cmap='coolwarm', alpha=0.3)
ax.contour(xx_x, yy_x, z_grid, levels=[0.5], colors=['#cf222e'], linewidths=[2.5], linestyles=['--'])

ax.scatter([0, 1], [0, 1], color='#0969da', s=160, edgecolors='black', lw=1.5, label='Classe 0 (Sortie 0)', zorder=5)
ax.scatter([0, 1], [1, 0], color='#cf222e', s=180, marker='^', edgecolors='black', lw=1.5, label='Classe 1 (Sortie 1)', zorder=5)

ax.set_title("MLP PyTorch : La Frontière Courbée Résout Enfin le XOR !", fontsize=12, fontweight='bold')
ax.set_xlabel("$x_1$", fontsize=11)
ax.set_ylabel("$x_2$", fontsize=11)
ax.legend(loc='upper right')
ax.grid(True, linestyle=':', alpha=0.5)
plt.show()


---
## 7 · Bonus Matériel : L'Accélération Apple Silicon Metal (MPS)

L'un des avantages cruciaux de PyTorch par rapport à NumPy est sa capacité à exécuter les calculs directement sur le processeur graphique :
- Sur PC / serveurs : **NVIDIA CUDA** (`device = "cuda"`).
- Sur les puces Apple Mac (M1, M2, M3, M4...) : **Metal Performance Shaders (MPS)** (`device = "mps"`).

Mesurons la différence de temps d'exécution sur une multiplication matricielle volumineuse ($3000 \times 3000$) entre le CPU et la puce Apple MPS :


In [ ]:
# Programme 16 — Benchmark CPU vs Apple Silicon MPS
import time

N = 3000

# 1. Calcul sur CPU
x_cpu = torch.randn(N, N, device="cpu")
# Warmup
_ = torch.matmul(x_cpu[:100, :100], x_cpu[:100, :100])

t0 = time.perf_counter()
res_cpu = torch.matmul(x_cpu, x_cpu)
t_cpu = time.perf_counter() - t0

# 2. Calcul sur GPU Apple MPS (si disponible)
if torch.backends.mps.is_available():
    device_mps = torch.device("mps")
    x_mps = torch.randn(N, N, device=device_mps)
    # Warmup MPS
    _ = torch.matmul(x_mps[:100, :100], x_mps[:100, :100])
    torch.mps.synchronize()

    t0 = time.perf_counter()
    res_mps = torch.matmul(x_mps, x_mps)
    torch.mps.synchronize()
    t_mps = time.perf_counter() - t0

    print("===============================================================")
    print(f"⚡ BENCHMARK MULTIPLICATION MATRICIELLE ({N} x {N})")
    print("===============================================================")
    print(f"  • Temps CPU PyTorch          : {t_cpu * 1000:.1f} ms")
    print(f"  • Temps GPU Apple Silicon MPS : {t_mps * 1000:.1f} ms")
    print(f"  🚀 Facteur d'accélération     : x{t_cpu / t_mps:.1f} plus rapide sur GPU !")
    print("===============================================================")
else:
    print(f"Temps CPU PyTorch : {t_cpu * 1000:.1f} ms (MPS non disponible)")


---
## 8 · Synthèse Générale & Aide-Mémoire (Cheat Sheet NumPy vs PyTorch)

Félicitations ! Vous venez de franchir le pont qui relie l'histoire du Machine Learning (1957) aux technologies modernes qui propulsent les plus grands modèles d'IA d'aujourd'hui.

### 📋 Tableau Comparatif : NumPy vs PyTorch

| Besoin / Concept | Approche NumPy (From Scratch) | Approche PyTorch (Moderne) |
|---|---|---|
| **Structure de données** | Tableau NumPy : `np.array` (CPU uniquement) | Tenseur PyTorch : `torch.Tensor` (CPU, CUDA, Apple MPS) |
| **Combinaison linéaire** | `z = X @ w + b` | `layer = nn.Linear(in_dim, out_dim)` |
| **Fonction d'activation** | `sigmoid = 1 / (1 + np.exp(-z))` | `nn.Sigmoid()`, `nn.ReLU()`, `nn.GELU()` |
| **Fonction de perte** | `bce = -np.mean(y*log(p) + ...)` | `criterion = nn.BCELoss()` |
| **Calcul des gradients** | Dérivation manuelle : `dw = (1/m) * X.T @ error` | **`loss.backward()` (Autograd automatique !)** |
| **Mise à jour des poids** | `w -= lr * dw` (SGD pur) | **`optimizer.step()` (`optim.Adam`, `SGD`)** |
| **Réseaux multicouches (MLP)** | Très complexe (chain rule manuelle) | `nn.Sequential(...)` en 4 lignes |
| **Résolution du XOR** | Impossible avec 1 neurone | Résolu en 6 lignes avec `nn.ReLU` |

---

### 🚀 Ce qu'il faut retenir :
1. **La différentiabilité est la clé de voûte de l'IA moderne** : sans fonctions lisses et dérivables, aucune descente de gradient n'est possible.
2. **Le gradient indique la pente, le signe moins assure la descente** : $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla L$.
3. **PyTorch décuple votre productivité** : grâce à **Autograd**, vous concevez librement n'importe quelle architecture sans jamais recalculer une dérivée à la main !
